# Übung 5: Lineare Optimierung in der Energiewirtschaft

**Ziel:** Lineare Optimierung verstehen: vom Tafelbeispiel per Hand bis zur
Day-Ahead-Optimierung eines Batteriespeichers.

## Lernpfad

1. **Theorie**: allgemeine Form eines linearen Programms (LP)
2. **Beispiel 1**: Cocktail-Bar (Tafelbeispiel, per Hand und mit Linopy)
3. **Beispiel 2**: Kraftwerkseinsatz mit Kohle, Gas und PV (Merit Order, Linopy)
4. **Beispiel 3**: Day-Ahead-Optimierung eines Batteriespeichers mit PyPSA
5. **Übungsaufgaben**: 4. Kraftwerk (Wind) · Heimspeicher mit echten 2025-Daten


In [ ]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import linopy
import pypsa


## 5.1 Theorie: Lineare Optimierung

### Allgemeine Form eines linearen Programms (LP)

$$\min_{x \geq 0}\ \underbrace{c^\top x}_{\text{Zielfunktion}}
\qquad \text{u.B.v.} \qquad
\underbrace{A\,x \leq b}_{\text{Nebenbedingungen}}$$

### Bestandteile

| Symbol | Name | Bedeutung |
|--------|------|-----------|
| $x \in \mathbb{R}^n_{\geq 0}$ | **Entscheidungsvektor** | Die $n$ Variablen, die der Optimierer frei wählen darf |
| $c \in \mathbb{R}^n$ | **Kostenvektor** (Zielfunktionskoeffizienten) | Gewichtung jeder Variable in der Zielfunktion; $c_j$ = Kosten pro Einheit $x_j$ |
| $c^\top x$ | **Zielfunktion** (Objektiv) | Skalarprodukt (der Wert, der minimiert wird) |
| $A \in \mathbb{R}^{m \times n}$ | **Restriktionsmatrix** | $A_{ij}$ beschreibt, wie viel Ressource $i$ eine Einheit von $x_j$ verbraucht |
| $b \in \mathbb{R}^m$ | **Rechte-Seite-Vektor** (RHS) | Verfügbare Kapazität/Obergrenze für jede der $m$ Nebenbedingungen |
| $A x \leq b$ | **Nebenbedingungen** | Jede Zeile $i$: $\sum_j A_{ij}\,x_j \leq b_i$; Ressourcenverbrauch $\leq$ Vorrat |

Maximierung wird auf Minimierung zurückgeführt: $\max c^\top x = -\min(-c)^\top x$


## 5.2 Beispiel 1: Cocktail-Bar *(vgl. Tafelbeispiel VL Prof. Zeiselmeier)*

### Aufgabenstellung

Sie verkaufen am Maifest drei Cocktails:

| Cocktail        | Preis    | Weißer Rum | Cointreau | Wodka  | Gin    |
|-----------------|----------|-----------:|----------:|-------:|-------:|
| Daiquiri        | 5,50 €   | 45 ml      | 30 ml     | 0 ml   | 0 ml   |
| Kamikaze        | 4,50 €   | 0 ml       | 30 ml     | 30 ml  | 0 ml   |
| Long Island ITT | 7,00 €   | 20 ml      | 20 ml     | 20 ml  | 20 ml  |

**Vorräte:** 5 l Rum, 6 l Cointreau, 4 l Wodka, 3 l Gin.

### Mathematische Formulierung

Entscheidungsvektor $x \in \mathbb{R}^3_{\geq 0}$ (Anzahl verkaufter Cocktails je Sorte).
Zielfunktion $z = c^\top x$ maximieren:

$$\max\ z = c^\top x \qquad \text{u.B.v.} \qquad A\,x \leq b,\quad x \geq 0$$

$$c = \begin{bmatrix} 5{,}5 \\ 4{,}5 \\ 7 \end{bmatrix},\quad
A = \begin{bmatrix} 45 & 0 & 20 \\ 30 & 30 & 20 \\ 0 & 30 & 20 \\ 0 & 0 & 20 \end{bmatrix},\quad
b = \begin{bmatrix} 5000 \\ 6000 \\ 4000 \\ 3000 \end{bmatrix}$$

### Lösung per Hand

**Schritt 1 — Gin ist exklusiv für Long Island:** $20\,x_3 \leq 3000 \Rightarrow x_3^\ast = 150$.

**Schritt 2 — Restproblem nach Abzug der Long-Island-Anteile:**
Wodka-Bedingung wird zu $30\,x_2 + 20\cdot 150 \leq 4000 \Rightarrow x_2^\ast \leq 33{,}\overline{3}$.
Rum-Bedingung wird zu $45\,x_1 + 20\cdot 150 \leq 5000 \Rightarrow x_1^\ast \leq 44{,}\overline{4}$.
Da der Preis pro Cocktail positiv ist, lohnt sich jeder zusätzlich verkaufte Cocktail — also Maximum nehmen.

**Schritt 3 — Kontrolle Cointreau:**
$30 \cdot 44{,}\overline{4} + 30 \cdot 33{,}\overline{3} + 20 \cdot 150 = 1333{,}3 + 1000 + 3000 = 5333{,}3$ ml.
Von 6000 ml verfügbar, also bleibt ein Rest von ca. 666 ml.

**Schritt 4 — Optimum:**

$$x^\ast = \begin{bmatrix} 44{,}\overline{4} \\ 33{,}\overline{3} \\ 150 \end{bmatrix},\quad
A\,x^\ast = \begin{bmatrix} 5000 \\ 5333{,}3 \\ 4000 \\ 3000 \end{bmatrix},\quad
b - A\,x^\ast = \begin{bmatrix} 0 \\ 666{,}7 \\ 0 \\ 0 \end{bmatrix}
\quad\Rightarrow\quad z^\ast = c^\top x^\ast \approx 1444{,}44\ €$$

Rum, Wodka und Gin sind **vollständig aufgebraucht** ($b - Ax^* = 0$). Cointreau ist die einzige Zutat mit Restmenge.

### Verbindung: Lineares Gleichungssystem (Gaußsche Elimination)

Was haben wir gerade gemacht? Wir wissen aus der Hand-Lösung, dass **drei der vier Vorräte exakt aufgebraucht** werden (Rum, Wodka, Gin). Diese drei Bedingungen liefern drei **Gleichungen** für drei Unbekannte — ein klassisches **LGS**, wie in Klasse 10 oder der Mathevorlesung:

$$A_B\,x^\ast = b_B
\qquad\text{mit}\qquad
A_B = \begin{pmatrix} 0 & 0 & 20 \\ 0 & 30 & 20 \\ 45 & 0 & 20 \end{pmatrix},\quad
b_B = \begin{pmatrix} 3000 \\ 4000 \\ 5000 \end{pmatrix}
\quad\text{(Zeilen: Gin, Wodka, Rum)}$$

Dieses System hat bereits **Dreiecksform** — reines Rückwärtseinsetzen (Gaußsche Elimination ohne Umformungsschritte):

$$\begin{align}
\text{I:}   \quad 0\cdot x_1 + 0\cdot x_2 + 20\cdot x_3  &= 3000 \\
\text{II:}  \quad 0\cdot x_1 + 30\cdot x_2 + 20\cdot x_3 &= 4000 \\
\text{III:} \quad 45\cdot x_1 + 0\cdot x_2 + 20\cdot x_3 &= 5000
\end{align}$$

Rückwärtseinsetzen (Zeile I hat nur $x_3$ → lösen, Ergebnis in II einsetzen → lösen, dann in III):

$$\begin{align}
\text{I:}   \quad 20\,x_3               &= 3000 \quad\Rightarrow\quad x_3^\ast = 150\\
\text{II:}  \quad 30\,x_2 + 20\cdot 150 &= 4000 \quad\Rightarrow\quad x_2^\ast = \tfrac{100}{3} \approx 33{,}\overline{3}\\
\text{III:} \quad 45\,x_1 + 20\cdot 150 &= 5000 \quad\Rightarrow\quad x_1^\ast = \tfrac{400}{9} \approx 44{,}\overline{4}
\end{align}$$

Das LP reduziert sich also auf ein LGS, sobald bekannt ist, welche Vorräte aufgebraucht werden. Der **Simplex-Algorithmus** macht genau das automatisch: er probiert systematisch verschiedene Auswahlen aus und findet diejenige, bei der das resultierende LGS die optimale Lösung liefert.


### In Linopy

In [ ]:
# Indexmenge: x_1 = Daiquiri, x_2 = Kamikaze, x_3 = Long Island ITT
cocktails = ["Daiquiri", "Kamikaze", "Long Island"]

# Preisvektor  c ∈ ℝ³   (c_j = Verkaufspreis pro Cocktail j in €)
c = pd.Series([5.5, 4.5, 7.0], index=cocktails)

# Restriktionsmatrix  A ∈ ℝ^{4×3}  (A[i,j] = ml Spirituose i pro Cocktail j)
A = pd.DataFrame(
    {"Daiquiri":    [45, 30,  0,  0],
     "Kamikaze":    [ 0, 30, 30,  0],
     "Long Island": [20, 20, 20, 20]},
    index=["Weißer Rum", "Cointreau", "Wodka", "Gin"],
)

# Rechte-Seite-Vektor  b ∈ ℝ⁴   (b_i = verfügbarer Vorrat Spirituose i in ml)
b = pd.Series({"Weißer Rum": 5000, "Cointreau": 6000, "Wodka": 4000, "Gin": 3000})

m = linopy.Model()

# lower=0 setzt die Nichtnegativität  x_j ≥ 0
# Die Obergrenzen kommen ausschließlich über A x ≤ b ins Modell
x = m.add_variables(lower=0, coords=[cocktails], dims=["cocktail"], name="x")

# Zielfunktion: Linopy kann direkt maximieren — kein Negieren nötig
m.add_objective((c * x).sum(), sense="max")

# Nebenbedingungen  A x ≤ b  (eine Zeile pro Spirituose)
for s in A.index:
    m.add_constraints((A.loc[s] * x).sum() <= b[s], name=s)

m.solve(solver_name="highs", log_to_console=False)

print(f"Maximaler Gewinn: {m.objective.value:.2f} EUR\n")
print(x.solution.to_pandas().round(2))


In [ ]:
# Tatsächlicher Verbrauch:  A x*
verbrauch = A @ x.solution.to_pandas()
pd.DataFrame({"Vorrat b (ml)":              b,
              "Verbraucht Ax* (ml)":        verbrauch.round(1),
              "Rest b−Ax* (ml)":            (b - verbrauch).round(1),
              "Vorrat aufgebraucht?":       (b - verbrauch).round(1) == 0})


## 5.3 Beispiel 2: Kraftwerkseinsatz (Merit Order)

### Aufgabenstellung

Ein Stadtwerk deckt den Bedarf über vier Stunden mit drei Kraftwerken zu minimalen Kosten.
Indexmengen: $T = \{1,2,3,4\}$ (Stunden), $I = \{\text{PV},\text{Kohle},\text{Gas}\}$ (Kraftwerke).

| Kraftwerk | Grenzkosten $c_i$ (€/MWh) | Max. Leistung $\bar p_{t,i}$ (MW) |
|-----------|---------------------------:|----------------------------------|
| PV        |  0                         | wetterabhängig (s. u.)           |
| Kohle     | 30                         | 80                               |
| Gas       | 60                         | 100                              |

Bedarf $d_t$ und verfügbare PV-Leistung je Stunde:

| $t$ | $d_t$ (MW) | $\bar p_{t,\text{PV}}$ (MW) |
|----:|-----------:|----------------------------:|
| 1   | 120        |  0                          |
| 2   |  90        | 30                          |
| 3   | 150        | 60                          |
| 4   | 110        | 20                          |

### Allgemeines Problem

Es ist wieder ein lineares Programm — gleiche Struktur wie Beispiel 1:

$$\min_{x \,\geq\, 0}\ c^\top x
\qquad\text{u.B.v.}\qquad A\,x \leq b$$

zusätzlich kommen Gleichungs-Nebenbedingungen vor (für die Bedarfsdeckung — kein Rest möglich, sonst fehlt Strom).

### Konkrete Formulierung

**Entscheidungsvektor** $P \in \mathbb{R}^{|T| \times |I|}_{\geq 0}$: Einspeiseleistung $p_{t,i} \geq 0$ [MW] je Stunde $t$ und Kraftwerk $i$.

$$\min_{P \,\geq\, 0}\; \sum_{t \in T}\sum_{i \in I} c_i\, p_{t,i}
\qquad\text{u.B.v.}\qquad
\begin{cases}
\displaystyle\sum_{i \in I} p_{t,i} = d_t & \forall\, t \in T \quad \text{(Nachfragedeckung)}\\[6pt]
p_{t,i} \leq \bar{p}_{t,i} & \forall\, t \in T,\; i \in I \quad \text{(Kapazitätsgrenzen)}
\end{cases}$$

$$c = \begin{bmatrix} 0 \\ 30 \\ 60 \end{bmatrix},\qquad
d = \begin{bmatrix} 120 \\ 90 \\ 150 \\ 110 \end{bmatrix},\qquad
\bar P = \begin{bmatrix} 0 & 80 & 100 \\ 30 & 80 & 100 \\ 60 & 80 & 100 \\ 20 & 80 & 100 \end{bmatrix}
\quad\small\text{(Zeilen: } t=1\ldots4\text{; Spalten: PV, Kohle, Gas)}$$

### Lösung per Hand: Merit Order

In jeder Stunde wird nach aufsteigenden Grenzkosten eingesetzt: PV (0 €) → Kohle (30 €) → Gas (60 €).

| $t$ | $d_t$ | PV  | Kohle | Gas | freie Gas-Kapazität (MW) | Kosten (€) |
|----:|------:|----:|------:|----:|-------------------------:|-----------:|
| 1   | 120   |   0 |    80 |  40 |  60                      | 4 800 |
| 2   |  90   |  30 |    60 |   0 | 100                      | 1 800 |
| 3   | 150   |  60 |    80 |  10 |  90                      | 3 000 |
| 4   | 110   |  20 |    80 |  10 |  90                      | 3 000 |
| **Σ** |     |     |       |     |                          | **12 600** |

Kohle läuft in jeder Stunde auf Maximalleistung — vollständig ausgelastet. Gas hat freie Kapazität — wird nur soweit nötig eingesetzt.

### Unterschiede zu Beispiel 1

| | Beispiel 1 (Cocktail) | Beispiel 2 (Kraftwerke) |
|---|---|---|
| **Variable** | $x_j$ — Anzahl Cocktails | $p_{t,i}$ — Leistung pro Stunde und Kraftwerk |
| **Index** | 1-D (Sorte $j$) | 2-D (Stunde $t$ × Kraftwerk $i$) |
| **Zielfunktion** | $\max\ c^\top x$ (Gewinn) | $\min \sum_{t,i} c_i\,p_{t,i}$ (Kosten) |
| **Kapazitäts-NB** | $A x \leq b$ (Vorrat) | $p_{t,i} \leq \bar p_{t,i}$ (Leistung) |
| **Deckungs-NB** | — | $\sum_i p_{t,i} = d_t$ als **Gleichung** |

Pro Stunde wird ein eigenes Gleichgewicht aufgestellt; die Stunden sind nur über den gemeinsamen Kostenvektor $c$ gekoppelt (Merit Order gilt zu jeder Stunde gleich).


### In Linopy

In [ ]:
# Indexmengen
T = [1, 2, 3, 4]          # T: Zeitschritte
I = ["PV", "Kohle", "Gas"] # I: Kraftwerke

# Grenzkostenvektor  c ∈ ℝ^|I|
c = pd.Series({"PV": 0, "Kohle": 30, "Gas": 60})
c.index.name = "i"         # Linopy braucht den Indexnamen zur Dim-Zuordnung bei p*c

# Bedarfsvektor  d ∈ ℝ^|T|
d = pd.Series([120, 90, 150, 110], index=T)

# Obere Schranken  P̄ ∈ ℝ^{|T|×|I|}
P_max = pd.DataFrame(
    {"PV":    [ 0, 30, 60, 20],
     "Kohle": [80, 80, 80, 80],
     "Gas":   [100,100,100,100]}, index=T,
)

m = linopy.Model()

# p_{t,i} ≥ 0, obere Schranke P̄_{t,i}
p = m.add_variables(lower=0, upper=P_max, coords=[T, I], dims=["t", "i"], name="p")

# min  Σ_{t,i} c_i · p_{t,i}  =  1_T^T P c
m.add_objective((p * c).sum())

# Σ_i p_{t,i} = d_t  ∀ t  (Nachfragedeckung)
for t in T:
    m.add_constraints(p.sel(t=t).sum() == d[t], name=f"d_t{t}")

m.solve(solver_name="highs", log_to_console=False)

dispatch = p.solution.to_pandas().round(1)
print(f"Gesamtkosten: {m.objective.value:,.0f} EUR\n")
print(dispatch)


In [ ]:
farben = {"PV": "#FFC107", "Kohle": "#5D4037", "Gas": "#FF7043"}

fig = go.Figure()
for kw in dispatch.columns:
    fig.add_bar(name=kw, x=dispatch.index, y=dispatch[kw], marker_color=farben[kw])
fig.add_scatter(x=T, y=d, name="Bedarf",
                mode="lines+markers", line=dict(color="black", dash="dash"))
fig.update_layout(barmode="stack", template="plotly_white",
                  title="Optimaler Kraftwerkseinsatz (Merit Order mit PV)",
                  xaxis_title="Stunde t", yaxis_title="Leistung (MW)",
                  height=380, margin=dict(t=50, b=40, l=50, r=20))
fig.show()


## 5.4 Beispiel 3: Day-Ahead-Optimierung eines Batteriespeichers (PyPSA)

[PyPSA](https://pypsa.readthedocs.io/) ist ein Framework für
Energiesystem-Optimierung. Das LP wird intern mit Linopy aufgebaut; wir
beschreiben das Modell auf der **Komponentenebene** (Bus, Generator, Last,
Speicher).

```
   Day-Ahead-Markt                Bus „electricity"        Batterie
   ┌─────────────┐  kaufen (p>0) ┌──────────────┐  laden  ┌──────────┐
   │  markt      │ ────────────► │              │ ──────► │          │
   │ (bidirekt., │               │              │  entl.  │ Storage  │
   │  marginal_  │ ◄──────────── │              │ ◄────── │          │
   │  cost=price)│ verkauf (p<0) └──────────────┘         └──────────┘
   └─────────────┘
```

Day-Ahead-Preise: synthetische 2024-Daten aus Übung 4 (`strompreise_2024.csv`),
Demo-Ausschnitt 14 Tage.

In [ ]:
df_da = pd.read_csv("../3_Zeitreihen/strompreise_2024.csv",
                    index_col=0, parse_dates=True).rename(columns={"price_eur_mwh": "price"})
df_da = df_da.loc["2024-06-01":"2024-06-14"]

bat = {"kapazitaet_mwh": 10.0, "p_max_mw": 5.0, "wirkungsgrad": 0.95, "soc0": 0.5}

n = pypsa.Network()
n.set_snapshots(df_da.index)
n.add("Bus", "electricity")

# Bidirektionaler Markt-Generator: p_min_pu=-1 erlaubt Verkauf, p_max_pu=1 Kauf.
# Bei marginal_cost=price minimiert das LP gleichzeitig Kaufkosten und maximiert
# Verkaufserlöse (negative Kosten = Erlös).
n.add("Generator", "markt", bus="electricity",
      p_nom=1e6,
      p_min_pu=-1.0, p_max_pu=1.0,
      marginal_cost=df_da["price"])

n.add("StorageUnit", "battery", bus="electricity",
      p_nom=bat["p_max_mw"],
      max_hours=bat["kapazitaet_mwh"] / bat["p_max_mw"],
      efficiency_store=bat["wirkungsgrad"], efficiency_dispatch=bat["wirkungsgrad"],
      state_of_charge_initial=bat["soc0"] * bat["kapazitaet_mwh"],
      cyclic_state_of_charge=False)
n.optimize(solver_name="highs", log_to_console=False)

# Marktleistung: positiv = Kauf, negativ = Verkauf (Konvention oben)
p_markt = n.generators_t.p["markt"]
kauf    = p_markt.clip(lower=0)
verkauf = (-p_markt).clip(lower=0)

# Batterie-Leistung: positiv = entladen (in Bus), negativ = laden (aus Bus)
p_bat  = n.storage_units_t.p["battery"]
laden  = (-p_bat).clip(lower=0)
entlad = p_bat.clip(lower=0)
soc    = n.storage_units_t.state_of_charge["battery"]

ladekosten     = (kauf  * df_da["price"]).sum()
entladeerloese = (verkauf * df_da["price"]).sum()
print(pd.Series({
    "Nettogewinn [€]":   entladeerloese - ladekosten,
    "Geladen [MWh]":     laden.sum(),
    "Entladen [MWh]":    entlad.sum(),
    "Ø Kaufpreis":       ladekosten / kauf.sum() if kauf.sum() > 0 else 0,
    "Ø Verkaufspreis":   entladeerloese / verkauf.sum() if verkauf.sum() > 0 else 0,
}).round(2).to_string())


In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=("Day-Ahead-Preis", "Lade-/Entladeleistung", "SOC"))

fig.add_scatter(x=df_da.index, y=df_da["price"], name="Preis",
                line=dict(color="#E64A19"), row=1, col=1)
fig.add_bar(x=df_da.index, y=entlad,  name="Entladen", marker_color="#43A047", row=2, col=1)
fig.add_bar(x=df_da.index, y=-laden,  name="Laden",    marker_color="#1E88E5", row=2, col=1)
fig.add_scatter(x=df_da.index, y=soc,  name="SOC",
                line=dict(color="#8E24AA"), fill="tozeroy", row=3, col=1)

fig.update_yaxes(title_text="€/MWh", row=1, col=1)
fig.update_yaxes(title_text="MW",    row=2, col=1)
fig.update_yaxes(title_text="MWh",   row=3, col=1)
fig.update_layout(template="plotly_white", height=620, hovermode="x unified",
                  barmode="overlay", margin=dict(t=50, b=40, l=50, r=20),
                  title="Batterie-Day-Ahead-Optimierung (synthet. 2024-Preise, 14 Tage)")
fig.show()

## 5.5 Übungsaufgaben

### Aufgabe 1: Viertes Kraftwerk im 4-Stunden-Beispiel

Erweitere das Modell aus **Beispiel 2** um ein viertes Kraftwerk:

| Kraftwerk | Grenzkosten (€/MWh) | Max. Leistung (MW) | Verfügbarkeit |
|-----------|--------------------:|-------------------:|---------------|
| **Wind**  | 0                   | 60                 | $\bar p^{\text{Wind}} = (40, 20, 10, 50)$ MW |

**Aufgaben:**

1. Erweitere das Linopy-Modell um Wind (variable Verfügbarkeit pro Stunde).
2. Berechne die neuen Gesamtkosten und vergleiche mit Beispiel 2 (12 600 €).
3. Plotte den Einsatzplan analog zur Vorlage als gestapeltes Säulendiagramm.
4. **Interpretation:** Welche Kraftwerke sind in welchen Stunden im Einsatz?
   Warum verdrängt Wind ausgerechnet Gas und nicht Kohle?

> *Hinweis:* Wind hat $c_\text{Wind} = 0$, also gleiche Grenzkosten wie PV; die
> Merit-Order setzt zuerst PV + Wind ein, dann Kohle, dann Gas.


In [ ]:
# Erweitere kraftwerke, grenzkosten und p_max um "Wind"
# (Verfügbarkeit [40, 20, 10, 50] MW, Grenzkosten 0 €/MWh).
# Baue das Linopy-Modell auf, optimiere und gib Gesamtkosten + Einsatzplan aus.

# Dein Code hier:


In [ ]:
# Plotte den Einsatzplan als gestapeltes Säulendiagramm (plotly).
# Bedarfslinie als gestrichelter Pfad oben drauf.

# Dein Code hier:


### Aufgabe 2: Heimspeicher der Studentin (echte 2025-Daten + Parameter-Variation)

In **Übung 3** habt ihr dynamische vs. statische Stromtarife verglichen und dabei
festgestellt, dass ein reiner Wechsel auf den dynamischen DA-Tarif für die Studentin
**CHR11 kaum Einsparung bringt**, denn ohne Flexibilität lässt sich der Preisvorteil
nicht nutzen.

Die naheliegende Folgefrage: **Lohnt sich ein kleiner Heimspeicher?**
Baut das PyPSA-Modell aus **Beispiel 3** nach, mit zwei Unterschieden:

| | Beispiel 3 | Aufgabe 2 |
|---|---|---|
| Netz | nur Speicher + Markt | Speicher + Haushaltslast (Load) |
| Tarif | reiner DA-Preis (Kauf & Verkauf) | DA + 200 €/MWh Retail-Aufschlag, **keine** Einspeisung |
| Zeitreihe | synthet. 2024-Daten, 1 h | CHR11-Daten 2025, **15 min** |

```
   DA-Markt                   Bus „haus"              Speicher
   ┌─────────────┐  kaufen   ┌───────────┐  laden   ┌──────────┐
   │ netz        │ ────────► │           │ ───────► │          │
   │ (DA+200 €)  │           │           │          │ StorageU │
   └─────────────┘           │           │  entl.   │          │
                             │           │ ◄─────── └──────────┘
                             └─────┬─────┘
                                   │ Last (CHR11)
                             ┌─────▼─────┐
                             │   Load    │
                             └───────────┘
```

**Daten aus Übung 3:**
- `dayahead_2025.csv`: Day-Ahead-Preise (15 min, Spalte `dayahead €/MWh`)
- `CHR11_last_15min_2025.csv`: Lastprofil (15 min, Spalte `last [kW]`)

Die Studentin bezieht Strom zu $p_\text{Haushalt}(t) = p_\text{DA}(t) + 20\ \text{ct/kWh}$
($= 200\ €/\text{MWh}$ Retail-Aufschlag).

**Aufgaben (alle für einen einzelnen Beispieltag, z. B. `2025-06-15`):**

1. Lade beide CSVs ein und merge sie zu einem DataFrame `df` mit **15-min-Index**
   (tz-naiv UTC). Filtere auf einen Beispieltag (z. B. `df.loc["2025-06-15"]`).
2. Schreibe eine Funktion `optimiere_heimspeicher(df_day, kapazitaet_kwh, leistung_kw, eta=0.95, retail_margin=200.0)` analog zu
   Beispiel 3, mit `Load`, `Generator` (nur Netzbezug, kein Verkauf) und
   `StorageUnit`. Nutze `n.set_snapshots(df_day.index, weightings_from_timedelta=True)`,
   damit die 15-min-Schritte automatisch mit Gewicht 0,25 h gerechnet werden.
   Die Funktion gibt das `Network`-Objekt zurück; die Tagesrechnung steht in `n.objective` (EUR/Tag).
3. **Referenz ohne Speicher** berechnen (Netzwerk **ohne** StorageUnit) und mit einem
   Standard-Speicher (5 kWh / 2,5 kW). Vergleiche die beiden Tagesrechnungen — wie groß ist die Einsparung?
4. **Parametervariation:** Rufe `optimiere_heimspeicher` in einer Schleife auf für
   Speichergrößen von $0{,}5,\ 1,\ 2,\ 4,\ 6,\ 8,\ 10$ kWh
   (C-Rate konstant: $p_\text{max} = \text{Kapazität} / 2\,\text{h}$).
   Plotte die Ersparnis in **€/Tag** und in **% der Tagesreferenzkosten** gegen die Kapazität.
5. **Wirtschaftlichkeit:** Nimm an die Speicherkosten betragen 200 €/kWh
   ([Preisvergleich Balkonkraftwerk-Speicher, idealo](https://www.idealo.de/preisvergleich/Liste/120478587/batteriespeicher-fuer-balkonkraftwerk.html)).
   Skaliere die Tagesersparnis aus Schritt 4 auf ein Jahr (× 365). Berechne pro Speichergröße:
   - Anschaffungskosten (200 €/kWh × Kapazität)
   - Jährliche Ersparnis (= Tagesersparnis × 365)
   - Amortisationszeit in Jahren (Anschaffungskosten / jährliche Ersparnis)
   Plotte oder tabelliere das Ergebnis. Ab welcher Speichergröße nimmt der Grenznutzen stark ab?
   Welcher Faktor begrenzt den maximalen Nutzen (Kapazität oder Leistung)?


In [ ]:
# Schritt 1: Daten laden (analog zum Daten-Einlesen in Beispiel 3)
# Lese dayahead_2025.csv und CHR11_last_15min_2025.csv ein.
# Zeitindex auf UTC normieren: .tz_convert("UTC").tz_localize(None)
# Merge zu einem DataFrame `df` mit Spalten "price" [€/MWh] und "load" [kW].

# Dein Code hier:


In [ ]:
# Schritt 2: Funktion optimiere_heimspeicher(df_day, kapazitaet_kwh, leistung_kw, eta=0.95, retail_margin=200.0)
#
# Baue das PyPSA-Netz analog zu Beispiel 3 auf:
#   n = pypsa.Network()
#   n.set_snapshots(df_day.index, weightings_from_timedelta=True)
#       # weightings_from_timedelta=True: PyPSA berechnet Δt aus dem Index automatisch
#       # bei 15-min-Daten: 0,25 h/Snapshot
#
#   n.add("Bus", "haus")
#   n.add("Load",  "last",  bus="haus", p_set=df_day["load"].values / 1e3)   # kW → MW
#   n.add("Generator", "netz", bus="haus",
#         marginal_cost=df_day["price"].values + retail_margin,
#         p_nom=1e6)                          # nur Bezug, keine Einspeisung
#   n.add("StorageUnit", "speicher", bus="haus",
#         p_nom=leistung_kw / 1e3,            # kW → MW
#         max_hours=kapazitaet_kwh / leistung_kw,
#         efficiency_store=eta, efficiency_dispatch=eta,
#         cyclic_state_of_charge=True)
#   n.optimize(solver_name="highs")
#   return n     # n.objective ist die Tagesrechnung in EUR
#
# Hinweis: Für die didaktische Schnelligkeit optimieren wir EINEN Tag
# (15-min-Auflösung → 96 Snapshots). Eine Jahresrechnung würde 35040 Snapshots
# bedeuten und mehrere Minuten dauern.

# Dein Code hier:


In [ ]:
# Schritt 4 — Parametervariation: [0.5, 1, 2, 4, 6, 8, 10] kWh, C-Rate konstant (p_max = kap / 2)
# Rufe optimize() in einer Schleife auf und plotte Ersparnis (€ und %) gegen Kapazität.

# Dein Code hier:
